<a href="https://colab.research.google.com/github/sruthi-kurra/llm-distillation/blob/main/01_teacher_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1 — Install dependencies

In [ ]:
!pip install transformers datasets evaluate rouge_score sentencepiece -q

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print("All dependencies installed!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
PyTorch version: 2.11.0+cu128
GPU available: True
All dependencies installed!


## Cell 2 — Load BART Large teacher model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading BART Large teacher model...")
teacher = AutoModelForSeq2SeqLM.from_pretrained(model_name)
teacher = teacher.cuda()
teacher.eval()

# Model size
total_params = sum(p.numel() for p in teacher.parameters())
print(f"\nTeacher model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Model size: {total_params / 1e6:.1f}M parameters")
print(f"Device: {next(teacher.parameters()).device}")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


Teacher model: facebook/bart-large-cnn
Total parameters: 406,290,432
Model size: 406.3M parameters
Device: cuda:0


 ## Cell 3 — Load CNN/DailyMail dataset

In [ ]:
from datasets import load_dataset

print("Loading CNN/DailyMail dataset...")
dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

print(f"\nDataset splits:")
print(f"Train: {len(dataset['train']):,} articles")
print(f"Validation: {len(dataset['validation']):,} articles")
print(f"Test: {len(dataset['test']):,} articles")

# Preview one example
example = dataset['test'][0]
print(f"\nExample article (first 300 chars):")
print(example['article'][:300])
print(f"\nReference summary:")
print(example['highlights'])

Loading CNN/DailyMail dataset...


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]


Dataset splits:
Train: 287,113 articles
Validation: 13,368 articles
Test: 11,490 articles

Example article (first 300 chars):
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou

Reference summary:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


## Cell 4 — Evaluate teacher ROUGE baseline

In [ ]:
import evaluate
import torch
from torch.utils.data import DataLoader

rouge = evaluate.load("rouge")

# Use 100 test examples for quick evaluation
test_subset = dataset['test'].select(range(100))

def generate_summary(article, max_input_length=1024, max_output_length=128):
    inputs = tokenizer(
        article,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        summary_ids = teacher.generate(
            inputs["input_ids"],
            num_beams=4,
            max_length=max_output_length,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Evaluating teacher on 100 test examples...")
predictions = []
references  = []

for i, example in enumerate(test_subset):
    pred = generate_summary(example['article'])
    predictions.append(pred)
    references.append(example['highlights'])
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/100...")

results = rouge.compute(predictions=predictions, references=references)

print(f"\n=== Teacher (BART Large) ROUGE Scores ===")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

# Save for comparison later
teacher_rouge = results
print("\nTeacher baseline saved!")

Evaluating teacher on 100 test examples...
Processed 10/100...
Processed 20/100...
Processed 30/100...
Processed 40/100...
Processed 50/100...
Processed 60/100...
Processed 70/100...
Processed 80/100...
Processed 90/100...
Processed 100/100...

=== Teacher (BART Large) ROUGE Scores ===
ROUGE-1: 0.3374
ROUGE-2: 0.1481
ROUGE-L: 0.2568

Teacher baseline saved!
